# Multimodal training v2: honest evaluation

Runs `training/train.py` from the repository. Nothing about the model is defined in this
notebook, so the trained weights always match the API code.

**What changed from v1**
- Paired text is *real* metadata (age, sex, X-ray view, lesion site), never generated from the label.
- Patient-disjoint splits: NIH official test list + Patient ID validation split, HAM10000 by `lesion_id`,
  brain MRI official Testing folder with training duplicates removed. Normal ("No Finding") X-rays are kept.
- One model, three heads trained together, so image-only and text-only API modes work.
- Temperatures and per-class thresholds fitted on validation; test evaluated once, with bootstrap CIs.

**Setup**
1. Accelerator: **GPU T4**. Internet: **on** (pretrained ImageNet / BioBERT weights).
2. Add input datasets: `nih-chest-xrays/data`, `kmader/skin-cancer-mnist-ham10000`,
   `sartajbhuvaji/brain-tumor-classification-mri`.
3. Run all. About 2.5 h with the defaults below; outputs land in `/kaggle/working/models`.

In [ ]:
REPO = "https://github.com/ARRY7686/Multi-Modal-Disease-Detection-System.git"
BRANCH = "main"   # set to the branch with training/ if it is not merged yet

!rm -rf /kaggle/working/repo
!git clone --depth 1 -b {BRANCH} {REPO} /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "-c",
    "import torch, transformers, sklearn; print(torch.__version__, transformers.__version__, "
    "sklearn.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')"],
    capture_output=True, text=True).stdout)
!ls /kaggle/input

## Train

Every `TrainSettings` field is a flag (`python -m training.train --help`). The main levers for
the time budget are `--chest-train-fraction` (share of NIH train/val *patients*) and `--epochs`.
For a quick end-to-end check first, add `--epochs 1 --max-steps-per-epoch 50 --chest-test-fraction 0.05 --bootstrap 50`.

In [ ]:
!python -m training.train \
    --data-root /kaggle/input \
    --out /kaggle/working/models \
    --chest-train-fraction 0.3 \
    --epochs 8 --frozen-epochs 1 \
    --batch-size 16 --num-workers 4

## Results

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/models/results.md").read()))

In [ ]:
import json
meta = json.load(open("/kaggle/working/models/multimodal_config.json"))
for h in meta["training_history"]:
    print(h["epoch"], "frozen" if h["frozen"] else "      ", f"loss {h['train_loss']:.4f}",
          {m: round(v, 4) for m, v in h["val_auc"].items()}, f"{h['seconds']:.0f}s")
print("temperatures:", meta["temperatures"])

## Package for the API

Download `models.zip` and unzip it into the repository's `models/` folder, or publish
`/kaggle/working/models` as a Kaggle dataset and use `scripts/download_models.py`.
Copy `results.md` into the README's Model Performance section.

In [ ]:
!cd /kaggle/working && zip -qr models.zip models && ls -lh models.zip models